# TP 1 — Méthodes d'Euler et transport numérique

**Analyse numérique — ESTP PGE1 S6 — 2025-2026**

---

## Consignes

- Ce notebook est **le sujet et le rendu**. Complétez les cellules de code marquées `# À COMPLÉTER` et répondez aux questions dans les cellules prévues.
- Chaque cellule peut être exécutée séparément en cliquant sur **Maj+Entrée**.
Lorsque vous lancer l'exécution d'une cellule, observez bien le symbole entre les crochets en haut à gauche. Si c'est une étoile, l'exécution est en cours !
- Le notebook doit **s'exécuter sans erreur** de bout en bout.
- Les réponses aux questions doivent être **argumentées** (2-3 phrases minimum).

## Barème

| Partie | Contenu | Points |
|--------|---------|--------|
| 1 | Euler explicite sur le café | /6 |
| 2 | Chute libre avec frottement | /3 |
| 3 | Transport d'un colorant (schéma décentré amont) | /6 |
| 4 | Instabilité et condition CFL | /2 |
| 5 | Pendule ou schéma implicite | /3 |
| **Total** | | **/20** |

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 12})

---

# Partie 1 — Euler explicite sur le café (/6 pts)

Au CM1, on a modélisé le refroidissement d'un café par l'EDO :

$$\frac{dT}{dt} = -k\,(T - T_{\text{amb}}), \qquad T(0) = T_0$$

avec $k = 0{,}1\;\text{min}^{-1}$, $T_0 = 90\;°\text{C}$, $T_{\text{amb}} = 20\;°\text{C}$.

La méthode d'Euler explicite approche la solution pas à pas :

$$y_{n+1} = y_n + \Delta t \, f(t_n,\, y_n)$$

## Exercice 1.1 — Implémenter Euler explicite (2 pts)

Compléter la fonction ci-dessous. Elle prend en entrée :
- `f` : la fonction $f(t, y)$ de l'EDO $y' = f(t, y)$
- `y0` : la condition initiale
- `t0`, `tf` : les bornes de l'intervalle de temps
- `dt` : le pas de temps

Elle renvoie deux tableaux NumPy : le premier pour stocker les instants $t_0, t_1, \ldots$ et le second pour stocker les valeurs approchées $y_0, y_1, \ldots$

In [2]:
def euler_explicite(f, y0, t0, tf, dt):
    """Méthode d'Euler explicite pour y' = f(t, y)."""
    n = int((tf - t0) / dt)
    t = np.zeros(n + 1)
    y = np.zeros(n + 1)
    t[0] = t0
    y[0] = y0

    for i in range(n):
        t[i + 1] = ...  # À COMPLÉTER
        y[i + 1] = ...  # À COMPLÉTER

    return t, y

Pour savoir si vous avez la bonne réponse, lancez la cellule suivante !

In [3]:
# --- Vérification automatique ---
# EDO test : y' = -y, y(0) = 1  =>  y(t) = exp(-t)
t_test, y_test = euler_explicite(lambda t, y: -y, 1.0, 0.0, 1.0, 0.01)
erreur_test = np.max(np.abs(y_test - np.exp(-t_test)))

print(f"Erreur max sur le test : {erreur_test:.6f}")
assert erreur_test < 0.01, "L'erreur semble trop grande, vérifiez votre formule."
print("✓ Test réussi !")

TypeError: float() argument must be a string or a real number, not 'ellipsis'

## Exercice 1.2 — Application au café (1 pt)

Définir la fonction `f_cafe(t, T)` correspondant à l'EDO du café, puis appeler `euler_explicite` avec $\Delta t = 2$ min sur l'intervalle $[0,\, 30]$ min.

**Rappel :** $f(t, T) = -k\,(T - T_{\text{amb}})$ avec $k = 0{,}1$ et $T_{\text{amb}} = 20$.

In [ ]:
k = 0.1
T0 = 90.0
T_amb = 20.0
dt = 2.0
tf = 30.0

def f_cafe(t, T):
    return ...  # À COMPLÉTER

t_euler, T_euler = ... # À COMPLÉTER

In [ ]:
# --- Tracé fourni ---
t_exact = np.linspace(0, tf, 300)
T_exact = T_amb + (T0 - T_amb) * np.exp(-k * t_exact)

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Solution exacte")
plt.plot(t_euler, T_euler, "o--", color="C0", label=f"Euler (Δt = {dt} min)")
plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Refroidissement du café — Euler vs exact")
plt.legend()
plt.grid(True)
plt.show()

## Exercice 1.3 — Influence du pas de temps (2 pts)

Le code ci-dessous trace la solution d'Euler pour plusieurs pas de temps. **Exécuter**, puis répondre : quel pas donne le résultat le plus proche de la solution exacte ? Que gagne-t-on en diminuant le pas ? Que perd-on ?

In [4]:
pas_a_tester = [5, 2, 1, 0.5]

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")

for dt_test in pas_a_tester:
    t_e, T_e = euler_explicite(f_cafe, T0, 0.0, tf, dt_test)
    plt.plot(t_e, T_e, "o--", markersize=4, label=f"Δt = {dt_test} min")

plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Influence du pas de temps")
plt.legend()
plt.grid(True)
plt.show()

NameError: name 't_exact' is not defined

<Figure size 900x400 with 0 Axes>

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 1.4 — Erreur en fonction du pas (1,5 pt)

Pour chaque pas $\Delta t$ dans la liste ci-dessous, calculer l'erreur maximale :

$$\max_i \left|T_{\text{exact}}(t_i) - T_i^{\text{Euler}}\right|$$

**À faire :**
1. Compléter la boucle : pour chaque valeur de `dt_test`, appeler `euler_explicite`, calculer la solution exacte aux mêmes instants, puis calculer l'erreur maximale.
2. Stocker chaque erreur dans la liste `erreurs`.

In [ ]:
pas_a_tester = [5, 2, 1, 0.5, 0.2, 0.1]
erreurs = []

for dt_test in pas_a_tester:
    # À COMPLÉTER :
    # 1. Appeler euler_explicite avec dt_test
    # 2. Calculer la solution exacte T_amb + (T0 - T_amb) * exp(-k * t) aux mêmes instants
    # 3. Calculer l'erreur maximale entre les deux
    # 4. Ajouter cette erreur à la liste erreurs
    pass

# --- Tracé fourni ---
plt.figure()
plt.loglog(pas_a_tester, erreurs, "o-", color="C1", linewidth=2)
plt.xlabel("Pas de temps Δt (min)")
plt.ylabel("Erreur maximale (°C)")
plt.title("Convergence de la méthode d'Euler")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

Pouvez-vous penser à d'autres façons de mesurer l'erreur d'approximation ? 

*(double-cliquez pour éditer)*

## Exercice 1.5 — Lire un graphe log-log (1 pt)

Le graphe ci-dessus utilise des **échelles logarithmiques** sur les deux axes. C'est un outil très courant en sciences de l'ingénieur pour mettre en évidence des **lois de puissance**.

Si l'erreur vérifie une loi du type $E = C \cdot (\Delta t)^p$, alors en prenant le logarithme :

$$\log E = p \cdot \log(\Delta t) + \log C$$

En échelle log-log, cette relation est une **droite de pente $p$**. La pente donne donc directement l'**ordre de convergence** de la méthode.

**À faire :**
1. Exécutez le code suivant utilisant `np.polyfit` et observez la pente. Interprêter ce que vous voyez.
2. Quel est l'ordre de convergence observé ? Est-ce cohérent avec ce qu'on attend d'Euler ?

In [ ]:
# Mesurer la pente en log-log
log_dt = np.log(pas_a_tester)
log_err = np.log(erreurs)

pente, ordonnee = np.polyfit(log_dt, log_err, 1)

print(f"Pente mesurée (ordre de convergence) : {pente:.2f}")

# Tracé avec la droite de régression
plt.figure()
plt.loglog(pas_a_tester, erreurs, "o", color="C1", markersize=8, label="Erreurs mesurées")
dt_reg = np.array(pas_a_tester)
plt.loglog(dt_reg, np.exp(ordonnee) * dt_reg**pente, "--", color="C1",
           label=f"Régression : pente = {pente:.2f}")
plt.xlabel("Pas de temps Δt (min)")
plt.ylabel("Erreur maximale (°C)")
plt.title("Convergence de la méthode d'Euler — échelle log-log")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*



## Question 1.6 — Que se passe-t-il si le pas est trop grand ? (0,5 pt)

Exécuter le code ci-dessous, puis répondre : le résultat obtenu avec $\Delta t = 15$ min est-il physiquement crédible ? Pourquoi ?

In [ ]:
t_big, T_big = euler_explicite(f_cafe, T0, 0.0, tf, 15.0)

plt.figure()
plt.plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")
plt.plot(t_big, T_big, "s-", color="red", linewidth=2, label="Euler (Δt = 15 min)")
plt.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
plt.xlabel("Temps (min)")
plt.ylabel("Température (°C)")
plt.title("Euler avec un pas trop grand")
plt.legend()
plt.grid(True)
plt.show()

print(f"T après 15 min : {T_big[1]:.1f} °C")

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 1.7 — Euler implicite (1 pt)

Au lieu d'évaluer la pente au temps $n$ (explicite), on peut l'évaluer au temps $n+1$ (implicite) :

$$T_{n+1} = T_n + \Delta t \left[-k\,(T_{n+1} - T_{\text{amb}})\right]$$

Le problème : $T_{n+1}$ apparaît des deux côtés ! Il faut isoler $T_{n+1}$ avant de pouvoir calculer.

**À faire :**
1. Au brouillon, résoudre l'équation donnant $T_{n+1}$ en fonction de $T_n$. 
2. Compléter la fonction `euler_implicite_cafe` ci-dessous.

In [ ]:
def euler_implicite_cafe(T0, T_amb, k, t0, tf, dt):
    """Euler implicite pour le modèle du café (formule isolée à la main)."""
    n = int((tf - t0) / dt)
    t = np.zeros(n + 1)
    T = np.zeros(n + 1)
    t[0] = t0
    T[0] = T0

    for i in range(n):
        t[i + 1] = ...  # À COMPLÉTER 
        T[i + 1] = ...  # À COMPLÉTER 

    return t, T

Pour savoir si votre Euler implicite est correct, lancez la cellule suivante ! *(Même idée que pour l'explicite : on compare à la solution exacte sur un cas test.)*

In [ ]:
# --- Vérification automatique — Euler implicite ---
# Même EDO que pour l'explicite : y' = -y, y(0) = 1  <=>  T' = -k(T - T_amb) avec k=1, T_amb=0, T(0)=1
# => T(t) = exp(-t)
t_imp_test, T_imp_test = euler_implicite_cafe(1.0, 0.0, 1.0, 0.0, 1.0, 0.01)
erreur_imp_test = np.max(np.abs(T_imp_test - np.exp(-t_imp_test)))

print(f"Erreur max sur le test (implicite) : {erreur_imp_test:.6f}")
assert erreur_imp_test < 0.01, "L'erreur semble trop grande, vérifiez votre formule."
print("✓ Test implicite réussi !")

**Votre démonstration :** *(double-cliquez pour éditer)*



## Exercice 1.8 — Explicite vs implicite : précision et robustesse (1 pt)

**a) À pas raisonnable.** Le code ci-dessous superpose les deux méthodes avec $\Delta t = 2$ min. Exécuter et observer : les résultats sont-ils très différents ?

**b) À pas excessif.** On relance avec $\Delta t = 15$ min. L'un des deux schémas diverge, l'autre survit. Lequel, et pourquoi ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_exact = np.linspace(0, 30, 300)
T_exact = T_amb + (T0 - T_amb) * np.exp(-k * t_exact)

# --- a) Pas raisonnable : dt = 2 min ---
dt_a = 2.0
t_exp, T_exp = euler_explicite(f_cafe, T0, 0.0, 30.0, dt_a)
t_imp, T_imp = euler_implicite_cafe(T0, T_amb, k, 0.0, 30.0, dt_a)

axes[0].plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")
axes[0].plot(t_exp, T_exp, "o--", color="C0", markersize=5, label=f"Explicite (Δt = {dt_a})")
axes[0].plot(t_imp, T_imp, "s--", color="C2", markersize=5, label=f"Implicite (Δt = {dt_a})")
axes[0].set_xlabel("Temps (min)")
axes[0].set_ylabel("Température (°C)")
axes[0].set_title("Pas raisonnable (Δt = 2 min)")
axes[0].legend()
axes[0].grid(True)

# --- b) Pas excessif : dt = 15 min ---
dt_b = 15.0
t_exp2, T_exp2 = euler_explicite(f_cafe, T0, 0.0, 30.0, dt_b)
t_imp2, T_imp2 = euler_implicite_cafe(T0, T_amb, k, 0.0, 30.0, dt_b)

axes[1].plot(t_exact, T_exact, "k-", linewidth=2, label="Exacte")
axes[1].plot(t_exp2, T_exp2, "o-", color="C0", markersize=7, label=f"Explicite (Δt = {dt_b})")
axes[1].plot(t_imp2, T_imp2, "s-", color="C2", markersize=7, label=f"Implicite (Δt = {dt_b})")
axes[1].axhline(y=T_amb, color="gray", linestyle=":", alpha=0.5)
axes[1].set_xlabel("Temps (min)")
axes[1].set_ylabel("Température (°C)")
axes[1].set_title("Pas excessif (Δt = 15 min)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Explicite  Δt=15 : T(15) = {T_exp2[1]:.1f} °C, T(30) = {T_exp2[2]:.1f} °C")
print(f"Implicite  Δt=15 : T(15) = {T_imp2[1]:.1f} °C, T(30) = {T_imp2[2]:.1f} °C")
print(f"Exacte           : T(15) = {T_amb + (T0-T_amb)*np.exp(-k*15):.1f} °C, T(30) = {T_amb + (T0-T_amb)*np.exp(-k*30):.1f} °C")

**Votre réponse :**

*a) À pas raisonnable, les deux schémas donnent-ils des résultats comparables ? L'un est-il clairement meilleur que l'autre ?*

*b) À pas excessif, que donne l'explicite ? Et l'implicite ? Lequel préféreriez-vous si vous deviez calculer vite avec un grand pas ?*

*(double-cliquez pour éditer)*



---

# Partie 2 — Chute libre avec frottement quadratique (/3 pts)

Un objet de masse $m$ tombe dans un fluide. Sa vitesse $v(t)$ vérifie :

$$\frac{dv}{dt} = g - \frac{c_f}{m}\,v(t)^2$$

avec $g = 9{,}81\;\text{m/s}^2$ et $c_f/m = 0{,}05\;\text{m}^{-1}$.

Contrairement au café (EDO linéaire), cette EDO est **non linéaire** à cause du terme $v^2$. La résolution exacte est plus complexe, mais la méthode d'Euler s'applique sans changement.

La **vitesse limite** est atteinte quand l'accélération s'annule : $v_{\lim} = \sqrt{\dfrac{g}{c_f/m}}$.

## Exercice 2.1 — Simuler la chute (1,5 pt)

**À faire :**
1. Définir la fonction `f_chute(t, v)` correspondant à l'EDO.
2. Appeler `euler_explicite` avec $v_0 = 0$, $\Delta t = 0{,}1$ s, sur l'intervalle $[0,\, 30]$ s.
3. Tracer $v(t)$ et superposer la vitesse limite théorique en trait pointillé horizontal. Inspirez-vous des codes précédents.

In [ ]:
g = 9.81
cf_sur_m = 0.05
v0 = 0.0
dt_chute = 0.1
tf_chute = 30.0

v_lim = np.sqrt(g / cf_sur_m)
print(f"Vitesse limite théorique : {v_lim:.2f} m/s")

def f_chute(t, v):
    return ...  # À COMPLÉTER question 1

t_chute, v_chute = ...  # À COMPLÉTER question 2

# À COMPLÉTER : tracer v(t) et la vitesse limite
# plt.figure()
# ...
# plt.show()

## Question 2.2 — Interprétation physique (1 pt)

1. La courbe $v(t)$ tend-elle bien vers $v_{\lim}$ ? À quel instant la vitesse atteint-elle environ 95 % de la vitesse limite ?
2. En quoi cette EDO diffère-t-elle de celle du café ? Euler a-t-il eu besoin d'être modifié pour la traiter ?

**Votre réponse :** *(double-cliquez pour éditer)*



## Question 2.3 — Influence du pas (0,5 pt)

Exécuter le code ci-dessous. Que se passe-t-il pour $\Delta t = 5$ s ? Est-ce que la solution reste physiquement raisonnable ?

In [ ]:
plt.figure()
plt.axhline(y=v_lim, color="gray", linestyle=":", label=f"v_lim = {v_lim:.1f} m/s")

for dt_test in [0.1, 0.5, 2, 5]:
    t_c, v_c = euler_explicite(f_chute, v0, 0.0, tf_chute, dt_test)
    plt.plot(t_c, v_c, "o--", markersize=3, label=f"Δt = {dt_test} s")

plt.xlabel("Temps (s)")
plt.ylabel("Vitesse (m/s)")
plt.title("Chute libre — influence du pas")
plt.legend()
plt.grid(True)
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Partie 3 — Transport d'un colorant (/6 pts)

Au CM2, on a modélisé le transport d'un colorant dans un canal par l'EDP :

$$\frac{\partial u}{\partial t} + c\,\frac{\partial u}{\partial x} = 0$$

avec $c = 2\;\text{m/s}$ (vitesse du courant). La solution exacte est une **translation pure** : $u(x,t) = u_0(x - ct)$.

En combinant Euler (temps) et différences finies rétrogrades (espace), on a obtenu le **schéma décentré amont** :

$$u_i^{n+1} = u_i^n - r\,(u_i^n - u_{i-1}^n), \qquad r = \frac{c\,\Delta t}{\Delta x}$$

## Exercice 3.1 — Implémenter le schéma décentré amont (2 pts)

Compléter la fonction ci-dessous. Elle renvoie un tableau 2D `U` de taille `(n_steps+1, M)` où `M` est le nombre de nœuds et `n_steps` le nombre de pas de temps.

**Rappel :** $u_i^{n+1} = u_i^n - r\,(u_i^n - u_{i-1}^n)$ avec $r = c\,\Delta t / \Delta x$.

In [ ]:
def transport_upwind(u0, c, dx, dt, n_steps):
    """Schéma décentré amont (upwind) pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    for n in range(n_steps):
        for i in range(1, M):
            U[n + 1, i] = ...  # À COMPLÉTER
        U[n + 1, 0] = U[n, 0]

    return U

In [ ]:
# --- Vérification automatique ---
# Avec r = 1, le créneau se translate exactement d'un noeud par pas de temps
u0_test = np.array([0, 0, 1, 1, 0, 0, 0, 0], dtype=float)
U_test = transport_upwind(u0_test, c=2.0, dx=2.0, dt=1.0, n_steps=1)

attendu = np.array([0, 0, 0, 1, 1, 0, 0, 0], dtype=float)
assert np.allclose(U_test[1], attendu), f"Résultat inattendu : {U_test[1]}"
print("✓ Test réussi (r = 1 : translation exacte d'un noeud) !")

## Exercice 3.2 — Retrouver les résultats du CM2 (1 pt)

Reproduire le calcul fait à la main en cours : canal de 14 m, 8 nœuds ($\Delta x = 2$ m), $\Delta t = 0{,}5$ s, $c = 2$ m/s, soit $r = 0{,}5$.

Condition initiale : concentration $u = 1$ g/L entre $x = 4$ m et $x = 6$ m, nulle ailleurs.

Vérifier que vos résultats correspondent au tableau du poly.

In [ ]:
c = 2.0
dx = 2.0
dt = 0.5

u0_cm2 = np.array([0, 0, 1, 1, 0, 0, 0, 0], dtype=float)
U_cm2 = transport_upwind(u0_cm2, c, dx, dt, n_steps=2)

# --- Affichage ---
print("Résultats (à comparer avec le tableau du CM2) :")
print()
print(f"{'n':>3} {'t (s)':>7}  " + "  ".join([f"  u_{i}" for i in range(8)]))
print("-" * 65)
for step in range(3):
    vals = "  ".join([f"{U_cm2[step, i]:5.2f}" for i in range(8)])
    print(f"{step:>3} {step * dt:>7.1f}  {vals}")

## Exercice 3.3 — Maillage fin et animation (1 pt)

On passe à un maillage fin (200 nœuds) pour mieux visualiser le phénomène. Le code ci-dessous appelle votre fonction `transport_upwind` et génère une **animation** du colorant se déplaçant dans le canal.

Exécuter et observer. L'animation peut prendre quelques secondes à s'afficher.

In [ ]:
L = 20.0
M = 200
c = 2.0
dx = L / (M - 1)
dt = 0.4 * dx / c
n_steps = 300

x = np.linspace(0, L, M)
u0_fin = np.where((x >= 2) & (x <= 4), 1.0, 0.0)

U_upwind = transport_upwind(u0_fin, c, dx, dt, n_steps)

# --- Animation fournie ---
fig, ax = plt.subplots()
line, = ax.plot(x, U_upwind[0, :], "C0", linewidth=2)
ax.set_xlim(0, L)
ax.set_ylim(-0.2, 1.4)
ax.set_xlabel("Position x (m)")
ax.set_ylabel("Concentration u (g/L)")
ax.set_title("Transport — décentré amont")
ax.grid(True)

def _update_upwind(frame):
    line.set_ydata(U_upwind[frame, :])
    ax.set_title(f"Transport — décentré amont — t = {frame * dt:.2f} s")

anim_upwind = FuncAnimation(
    fig, _update_upwind,
    frames=range(0, n_steps + 1, 3), interval=50)
plt.close(fig)
HTML(anim_upwind.to_jshtml())

## Exercice 3.4 — Comparaison avec la solution exacte (0,5 pt)

La solution exacte est une translation pure : le créneau se déplace à vitesse $c$ sans se déformer. Le code ci-dessous superpose la solution numérique et la solution exacte au temps final.

In [ ]:
t_final = n_steps * dt
u_exact_final = np.where(
    (x >= 2 + c * t_final) & (x <= 4 + c * t_final), 1.0, 0.0)

plt.figure()
plt.plot(x, u_exact_final, "k--", linewidth=2, label="Solution exacte (translation)")
plt.plot(x, U_upwind[-1, :], "C0", linewidth=2, label="Schéma décentré amont")
plt.plot(x, u0_fin, ":", color="gray", alpha=0.5, label="Condition initiale")
plt.xlabel("Position x (m)")
plt.ylabel("Concentration u (g/L)")
plt.title(f"Comparaison à t = {t_final:.1f} s")
plt.legend()
plt.grid(True)
plt.show()

## Question 3.5 — Diffusion numérique (1 pt)

Le profil numérique est-il une translation exacte du créneau initial ? Quel défaut observez-vous ? Comment ce défaut évoluerait-il si l'on prenait un maillage plus fin ?

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 3.6 — Mesurer la diffusion numérique (1,5 pt)

On veut quantifier la diffusion numérique au lieu de simplement l'observer. Pour cela, on mesure l'**erreur $L^2$** entre la solution numérique et la solution exacte :

$$E^n = \sqrt{\Delta x \sum_{i} \left(u_i^n - u_{\text{exact}}(x_i, t^n)\right)^2}$$

**À faire :**
1. Compléter le code ci-dessous pour calculer l'erreur $L^2$ à chaque pas de temps sauvegardé.
2. Exécuter et observer : l'erreur croît-elle au cours du temps ?
3. Refaire le calcul avec un maillage deux fois plus fin (`M = 400`, en adaptant `dt` pour garder $r = 0{,}4$) et superposer les deux courbes d'erreur.
4. Répondre : le raffinement du maillage réduit-il l'erreur ? C'est ce qu'on appelle la **convergence en espace**.

In [ ]:
# --- Maillage de référence (M = 200) ---
L = 20.0
M = 200
c = 2.0
dx = L / (M - 1)
dt_transport = 0.4 * dx / c
n_steps = 300

x = np.linspace(0, L, M)
u0_fin = np.where((x >= 2) & (x <= 4), 1.0, 0.0)

U_upwind = transport_upwind(u0_fin, c, dx, dt_transport, n_steps)

# Calcul de l'erreur L2 à chaque pas de temps
pas_sauvegarde = range(0, n_steps + 1, 10)
temps_sauv = []
erreurs_L2 = []

for n in pas_sauvegarde:
    t_n = n * dt_transport
    # Solution exacte : créneau translaté de c * t_n
    u_exact_n = np.where((x >= 2 + c * t_n) & (x <= 4 + c * t_n), 1.0, 0.0)
    err = ...  # À COMPLÉTER : norme L2 = sqrt(dx * sum((U_upwind[n,:] - u_exact_n)**2))
    temps_sauv.append(t_n)
    erreurs_L2.append(err)

# --- À COMPLÉTER : refaire avec M = 400 ---
# M2 = 400
# dx2 = L / (M2 - 1)
# dt2 = 0.4 * dx2 / c   # on garde r = 0.4
# n_steps2 = int(n_steps * dt_transport / dt2)  # même durée totale
# ...

# --- Tracé ---
plt.figure()
plt.plot(temps_sauv, erreurs_L2, "C0", linewidth=2, label=f"M = {M}")
# plt.plot(temps_sauv2, erreurs_L2_2, "C1", linewidth=2, label=f"M = {M2}")  # décommenter
plt.xlabel("Temps (s)")
plt.ylabel("Erreur L²")
plt.title("Évolution de l'erreur de diffusion numérique")
plt.legend()
plt.grid(True)
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Partie 4 — Instabilité et condition CFL (/2 pts)

Au CM2, on a vu qu'en remplaçant la différence rétrograde par la **différence centrée**, le schéma devient :

$$u_i^{n+1} = u_i^n - \frac{r}{2}\,(u_{i+1}^n - u_{i-1}^n)$$

Ce schéma, pourtant d'apparence naturelle, produit des résultats catastrophiques.

## Exercice 4.1 — Implémenter le schéma centré (1 pt)

Compléter la fonction ci-dessous. La seule différence avec `transport_upwind` est la formule de mise à jour.

**Rappel :** $u_i^{n+1} = u_i^n - \dfrac{r}{2}\,(u_{i+1}^n - u_{i-1}^n)$.

In [ ]:
def transport_centre(u0, c, dx, dt, n_steps):
    """Schéma centré explicite pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    for n in range(n_steps):
        for i in range(1, M - 1):
            U[n + 1, i] = ...  # À COMPLÉTER
        U[n + 1, 0] = U[n, 0]
        U[n + 1, -1] = U[n, -1]

    return U

## Exercice 4.2 — Observer l'instabilité (1 pt)

Le code ci-dessous anime **côte à côte** le schéma décentré amont et le schéma centré, sur les mêmes données que la partie 3. Exécuter et observer attentivement.

In [ ]:
n_compare = 60
U_upwind_short = transport_upwind(u0_fin, c, dx, dt, n_compare)
U_centre_short = transport_centre(u0_fin, c, dx, dt, n_compare)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

line1, = axes[0].plot(x, U_upwind_short[0, :], "C0", linewidth=2)
axes[0].set_xlim(0, L)
axes[0].set_ylim(-3, 4)
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("u (g/L)")
axes[0].set_title("Décentré amont")
axes[0].grid(True)
axes[0].axhline(y=0, color="gray", linestyle=":", alpha=0.4)

line2, = axes[1].plot(x, U_centre_short[0, :], "C3", linewidth=2)
axes[1].set_xlim(0, L)
axes[1].set_xlabel("x (m)")
axes[1].set_title("Centré")
axes[1].grid(True)
axes[1].axhline(y=0, color="gray", linestyle=":", alpha=0.4)

def _update_compare(frame):
    line1.set_ydata(U_upwind_short[frame, :])
    axes[0].set_title(f"Décentré amont — t = {frame * dt:.2f} s")
    line2.set_ydata(U_centre_short[frame, :])
    axes[1].set_title(f"Centré — t = {frame * dt:.2f} s")

anim_compare = FuncAnimation(
    fig, _update_compare,
    frames=range(0, n_compare + 1, 2), interval=150)
plt.close(fig)
HTML(anim_compare.to_jshtml())

## Question 4.3 — Analyse de l'instabilité (1 pt)

Comparez les deux animations. Qu'observez-vous pour le schéma centré (valeurs négatives, oscillations) ? Pourquoi un schéma qui utilise une approximation « plus symétrique » de la dérivée produit-il un résultat pire que le schéma décentré ?

**Votre réponse :** *(double-cliquez pour éditer)*



## Exercice 4.4 — Condition CFL (0,5 pt)

Même le schéma décentré amont peut devenir instable si le rapport $r = c\,\Delta t / \Delta x$ est trop grand. Le code ci-dessous teste trois valeurs de $r$. Exécuter et identifier la valeur critique.

In [ ]:
valeurs_r = [0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, r_val in enumerate(valeurs_r):
    dt_cfl = r_val * dx / c
    n_cfl = int(3.0 / dt_cfl)
    U_cfl = transport_upwind(u0_fin, c, dx, dt_cfl, n_cfl)

    axes[idx].plot(x, u0_fin, "k--", alpha=0.4, label="t = 0")
    axes[idx].plot(
        x, U_cfl[-1, :], "C0", linewidth=2,
        label=f"t = {n_cfl * dt_cfl:.1f} s")
    axes[idx].set_xlim(0, L)
    axes[idx].set_ylim(-1.5, 2.5)
    axes[idx].set_xlabel("x (m)")
    axes[idx].set_title(f"r = {r_val}")
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True)

plt.suptitle("Schéma décentré amont — influence de r", fontsize=14)
plt.tight_layout()
plt.show()

## Question 4.5 — Interpréter la condition CFL (1 pt)

Pour le schéma décentré amont, quelle condition doit vérifier $r$ pour que le schéma reste stable ? Que se passe-t-il concrètement si l'on raffine l'espace ($\Delta x$ plus petit) sans adapter $\Delta t$ ?

**Votre réponse :** *(double-cliquez pour éditer)*



---

# Partie 5 — Approfondissements (bonus, jusqu'à 3 pts)

Cette partie comporte deux exercices indépendants. Traiter **un au choix**.

## Exercice 5.1 — Pendule simple et conservation de l'énergie (3 pt)

Utiliser Euler pour simuler un pendule simple :

$$\begin{cases} \theta'(t) = v(t) \\ v'(t) = -\omega_0^2 \sin(\theta(t)) \end{cases}$$

avec $\omega_0 = 1{,}5\;\text{rad/s}$, $\theta(0) = 0{,}8\;\text{rad}$, $v(0) = 0$.

**À faire :**
1. Compléter la boucle d'Euler pour le **système de deux équations**.
2. Tracer $\theta(t)$ et le portrait de phase $(\theta, v)$.
3. Calculer et tracer l'énergie mécanique $E(t) = \frac{1}{2}v^2 - \omega_0^2 \cos(\theta)$ au cours du temps.

**Question :** L'énergie est-elle conservée par le schéma d'Euler ? Que signifie physiquement la dérive observée ?

In [ ]:
omega0 = 1.5
h_pend = 0.02
n_pendule = 2000

t_pend = np.zeros(n_pendule + 1)
theta = np.zeros(n_pendule + 1)
v_pend = np.zeros(n_pendule + 1)

theta[0] = 0.8
v_pend[0] = 0.0

for i in range(n_pendule):
    t_pend[i + 1] = t_pend[i] + h_pend
    theta[i + 1] = ...   # À COMPLÉTER
    v_pend[i + 1] = ...  # À COMPLÉTER

# --- Tracés ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(t_pend, theta)
axes[0].set_xlabel("t (s)")
axes[0].set_ylabel("θ (rad)")
axes[0].set_title("Angle du pendule")
axes[0].grid(True)

axes[1].plot(theta, v_pend)
axes[1].set_xlabel("θ (rad)")
axes[1].set_ylabel("v (rad/s)")
axes[1].set_title("Portrait de phase")
axes[1].grid(True)

# À COMPLÉTER : calculer et tracer l'énergie mécanique
# E = 0.5 * v_pend**2 - omega0**2 * np.cos(theta)
# axes[2].plot(t_pend, E)
# axes[2].set_xlabel("t (s)")
# axes[2].set_ylabel("E")
# axes[2].set_title("Énergie mécanique")
# axes[2].grid(True)

plt.tight_layout()
plt.show()

**Votre réponse sur la conservation de l'énergie :** *(double-cliquez pour éditer)*



## Exercice 5.2 — Schéma implicite pour le transport (3 pt)

Le schéma implicite décentré amont s'écrit :

$$(1 + r)\,u_i^{n+1} - r\,u_{i-1}^{n+1} = u_i^n$$

Cela revient à résoudre un système linéaire **bidiagonal** à chaque pas de temps.

**À faire :**
1. Compléter la construction de la matrice et la résolution (utiliser `np.linalg.solve`).
2. Tester avec $r = 1{,}5$ : le schéma explicite explose, le schéma implicite survit.
3. Tracer **sur le même graphe** le résultat implicite et le résultat explicite pour $r = 1{,}5$.

**Question :** Pourquoi le schéma implicite est-il plus stable ? Quel est son inconvénient par rapport à l'explicite ?

In [ ]:
def transport_implicite(u0, c, dx, dt, n_steps):
    """Schéma implicite décentré amont pour l'équation de transport."""
    M = len(u0)
    r = c * dt / dx
    U = np.zeros((n_steps + 1, M))
    U[0, :] = u0.copy()

    A = np.eye(M) * (1 + r) + np.eye(M, k=-1) * (-r)
    A[0, 0] = 1.0
    A[0, 1] = 0.0

    for n in range(n_steps):
        rhs = U[n, :].copy()
        U[n + 1, :] = ...  # À COMPLÉTER : résoudre A @ U[n+1,:] = rhs

    return U

# --- Comparaison explicite vs implicite pour r = 1.5 ---
L = 20.0
M = 200
c = 2.0
dx = L / (M - 1)
x = np.linspace(0, L, M)
u0_fin = np.where((x >= 2) & (x <= 4), 1.0, 0.0)

dt_cfl = 1.5 * dx / c
n_imp = int(3.0 / dt_cfl)

U_exp_15 = transport_upwind(u0_fin, c, dx, dt_cfl, n_imp)
U_imp_15 = transport_implicite(u0_fin, c, dx, dt_cfl, n_imp)

# À COMPLÉTER : tracer sur le même graphe
# - la condition initiale (trait pointillé gris)
# - le résultat explicite (qui explose)
# - le résultat implicite (qui survit)
plt.figure()
plt.plot(x, u0_fin, "k--", alpha=0.4, label="t = 0")
# ...
plt.xlabel("x (m)")
plt.ylabel("u (g/L)")
plt.title("Explicite vs Implicite — r = 1.5")
plt.legend()
plt.grid(True)
plt.show()

**Votre réponse :** *(double-cliquez pour éditer)*

